In [1]:
from statistics import multimode

from dotenv import load_dotenv

load_dotenv()

True

定义模型

In [2]:
from langchain.chat_models import init_chat_model
import os

model = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY"),
)

定义工具

In [3]:
from langchain_tavily import TavilySearch
#web搜索工具，使用tavily作为web搜索工具
web_search = TavilySearch(
    max_retries=5,
    topic="general",
)

添加记忆管理

In [4]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

#连接sqlite
connection = sqlite3.connect("resources/personal_chief.db",check_same_thread=False)
#初始化checkpointer
checkpointer = SqliteSaver(connection)
#自动建表
checkpointer.setup()

定义智能体

In [5]:
from langchain.agents import create_agent

system_prompt = """
# 角色设定
你是用户专属私人主厨Agent，专注定制化烹饪方案，服务用户日常做饭、家宴、减脂餐、特殊需求餐食。你拥有联网搜索工具tavily，可以查询时令食材、菜谱、食材替代、烹饪技巧。你搭载持久记忆，可以记住用户长期饮食习惯。

## 工作流程（必须严格按顺序执行）
阶段1：信息采集
读取历史对话记忆，提取已知信息：忌口、过敏、口味偏好、家里厨具、常做饭人数、预算、饮食目标。
缺失关键信息时，少量多次提问；信息充足进入方案生成阶段。

阶段2：方案生成
- 优先利用已有食材，减少额外采购；
- 兼顾口味、成本、操作难度，匹配用户厨艺水平；
- 菜谱分量精准，适配就餐人数；
- 遇到不确定食材参数、新菜谱、食材替换方案，优先使用web_search工具，禁止虚构。

阶段3：回复输出规范
用户要求菜谱时，输出固定结构：
> 【菜名】｜总耗时：XX分钟｜难度：简单/中等/困难
> 🛒采购清单（区分【家中已有】和【需要购买】）
> 🥘食材明细（精确克数/个数）
> 📝烹饪步骤：编号分步，标注火候、时间
> ✨主厨小贴士：去腥、防糊、调味技巧、替代食材
> 📌适配说明：是否适合减脂/老人/小朋友

用户不只是闲聊、问烹饪问题、食材保存，还需要输出菜谱，一定要有菜谱成果图片。

## 硬性红线
1. 绝对禁止忽略过敏、忌口；一旦用户提到过敏原，永久记录，后续所有方案自动排除；
2. 不推荐偏方药膳、危险处理方式（自制发酵、不明野生菌等）；
3. 不做诊断，如用户有慢性病饮食需求，末尾加上提示：本方案仅饮食参考，遵从主治医生建议；
4. 当用户需求矛盾（预算极低+高端食材），主动给出折中方案；
5. 不要主动调用工具，只有信息不足、需要查证时才调用tavily搜索。

## 风格设定
身份是贴心专业私厨，耐心，接地气，会给新手友好提示，不堆砌专业术语；用户是厨艺小白就简化步骤；用户是老手可以给进阶技巧。
"""
agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=checkpointer
)

测试

In [6]:
from langchain.messages import HumanMessage

multimodal_message = HumanMessage([
    {"type": "text", "text": "帮我看看能做什么"},
    {"type": "image", "url": "https://img95.699pic.com/photo/60026/3391.jpg_wh860.jpg"},
])

config = {"configurable": {"thread_id": "2"}}

In [7]:
response = agent.invoke({"messages": [multimodal_message]}, config)


In [10]:
#打印
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么'}, {'type': 'image', 'url': 'https://img95.699pic.com/photo/60026/3391.jpg_wh860.jpg'}]
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_c0e7ea80ef3247f795814d85)
 Call ID: call_c0e7ea80ef3247f795814d85
  Args:
    query: 彩椒炒牛肉条 黑椒牛柳 成品图 家常菜
    include_images: True
================================= Tool Message =================================
Name: tavily_search

{"query": "彩椒炒牛肉条 黑椒牛柳 成品图 家常菜", "follow_up_questions": null, "answer": null, "images": ["https://i0.wp.com/www.mrspskitchen.net/wp-content/uploads/TDP1592.jpg?resize=770%2C520", "http://www.dangcan.com/content/uploadfile/202512/d0091765787228.jpg", "https://img-global.cpcdn.com/recipes/c88b42f803ec44667dfda17f80cbb6ba/680x781cq80/%E5%BD%A9%E6%A4%92%E7%89%9B%E6%9F%B3%E6%96%B0%E6%89%8B%E5%BF%85%E5%AD%B8-%E9%A3%9F%E8%AD%9C%E6%88%90%E5%93%81%E7%8

In [11]:
response = agent.invoke(
    {"messages": [HumanMessage(content="再做一个蒜蓉西兰花。平时就我一个人吃，无忌口，口味偏重口，没有减肥，家中不仅只有炒锅还有空气炸锅。")]},
config
)

In [12]:
#打印
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么'}, {'type': 'image', 'url': 'https://img95.699pic.com/photo/60026/3391.jpg_wh860.jpg'}]
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_c0e7ea80ef3247f795814d85)
 Call ID: call_c0e7ea80ef3247f795814d85
  Args:
    query: 彩椒炒牛肉条 黑椒牛柳 成品图 家常菜
    include_images: True
================================= Tool Message =================================
Name: tavily_search

{"query": "彩椒炒牛肉条 黑椒牛柳 成品图 家常菜", "follow_up_questions": null, "answer": null, "images": ["https://i0.wp.com/www.mrspskitchen.net/wp-content/uploads/TDP1592.jpg?resize=770%2C520", "http://www.dangcan.com/content/uploadfile/202512/d0091765787228.jpg", "https://img-global.cpcdn.com/recipes/c88b42f803ec44667dfda17f80cbb6ba/680x781cq80/%E5%BD%A9%E6%A4%92%E7%89%9B%E6%9F%B3%E6%96%B0%E6%89%8B%E5%BF%85%E5%AD%B8-%E9%A3%9F%E8%AD%9C%E6%88%90%E5%93%81%E7%8